# SEMEFO-DF: Bitácora onomástica de ingresos

Limpieza inicial de datos.
El principal objetivo es generar las propuestas de catálogo y determinar que limpiezas serán necesarias.





In [1]:
import pandas as pd
from numpy import nan
from warnings import warn

from helper_pandas import flatten_list, simplify_string, astype_and_separate_error, replace_edades_menor_al_año
from helper_pandas import join_strings_with_nans, punctuation_in_string, failures_cast_to_type
from helper_pandas import replace_by_column, list_to_nestdic, create_hierarchical_catalog

def detect_sd(s, replace_value=nan, substring=False):
    str_sd = ['S-d', 's-d', 'S/n', 'S-D', 'S/N', '?', ]
    if s in str_sd:
        return replace_value
    if type(s)==str and len(s.strip())==0:
        return replace_value
    return s


In [2]:
file_raw = "data_raw/SEMEFO_220908/1974_SEMEFO-DF_Bitácora onomástica de ingreso_David.xlsx"
file_col = "data_raw/SEMEFO_220908/SEMEFO_mapeo_headers.csv"
file_replace = "data_raw/SEMEFO_220908/SEMEFO_1974_correciones.csv"
file_clean = "data_clean/1974_SEMEFO-DF_Bitácora onomástica de ingreso_David_clean.csv"
file_profi = "profiles/1974_SEMEFO-DF_Bitácora onomástica de ingreso_David_clean.html"

In [3]:
df = pd.read_excel(file_raw, dtype=str)
df.dtypes

Apellido Paterno                  object
Apellido Materno                  object
Primer Nombre                     object
Segundo Nombre                    object
Tercer nombre                     object
Alternativo                       object
NP                                object
NOMBRE                            object
PROCEDENCIA (Agencia de MP)       object
Número de  Averiguación Previa    object
FOLIO EXP SEMEFO                  object
SEXO                              object
FECHA                             object
EDAD                              object
DIAGNÓSTICO                       object
BITÁCORA DE INGRESOS              object
PÁGINA PDF                        object
FOJA                              object
POSIBLE IDENTIFICADO COMO         object
OBSERVACIONES CAPTURISTA          object
Restos                            object
Unnamed: 21                       object
dtype: object

## Revisar columnas

In [4]:
# Limpiar nombres propios
col_nom = ['Primer Nombre', 'Segundo Nombre', 'Tercer nombre']
df['NOMBRES'] = df[col_nom].apply(join_strings_with_nans, axis=1)
df = df.drop(col_nom, axis=1)

# Change col names to standard
with open(file_col, 'r') as f:
    dic_cols = pd.read_csv(f)
    all_cols = list(dic_cols['Estandar'].unique())
    dic_cols = dict(zip(dic_cols['Original'], dic_cols['Estandar']))
dic_cols, all_cols

df.columns = [simplify_string(c, str_case='upper') for c in df.columns]
df.columns = [dic_cols[c] if c in dic_cols.keys() else c for c in df.columns]
df.columns

# Add missing, remove unwanted, and sort
for col in all_cols:
    if col not in df.columns:
        warn('Missing expected column in file. Adding empty column {}'.format(col))
        df[col] = nan
for col in df.columns:
    if df[col].isnull().all():
        warn('Empty column {}'.format(col))
    if col not in all_cols:
        warn('Unexpected expected column in file. Removing column {}'.format(col))
        df = df.drop(col, axis=1)
df = df[all_cols]
df.dtypes

/tmp/ipykernel_19186/21314182.py:24: UserWarning: Empty column NUM PROG
  warn('Empty column {}'.format(col))
/tmp/ipykernel_19186/21314182.py:24: UserWarning: Empty column BITACORA DE INGRESOS
  warn('Empty column {}'.format(col))
/tmp/ipykernel_19186/21314182.py:26: UserWarning: Unexpected expected column in file. Removing column UNNAMED 21
  warn('Unexpected expected column in file. Removing column {}'.format(col))


NUM PROG                     object
NOMBRE COMPLETO              object
APELLIDO PATERNO             object
APELLIDO MATERNO             object
NOMBRES PROPIOS              object
NOMBRE ALTERNATIVO           object
PROCEDENCIA MP Y DEL         object
AVERIGUACION PREVIA          object
EXP SEMEFO                   object
FECHA                        object
EDAD                         object
SEXO                         object
DIAGNOSTICO                  object
BITACORA DE INGRESOS         object
PAGINA PDF                   object
FOJA                         object
POSIBLE IDENTIFICADO COMO    object
OBSERVACIONES CAPTURISTA     object
TIPOS DE RESTOS              object
dtype: object

In [5]:
col_join = 'OBSERVACIONES CAPTURISTA'
col_delete = ['NOMBRE ALTERNATIVO', 'POSIBLE IDENTIFICADO COMO']
df[col_join] = df[[col_join]+col_delete].apply(join_strings_with_nans, axis=1, sep='\n')
df = df.drop(col_delete, axis=1)
df.dtypes

NUM PROG                    object
NOMBRE COMPLETO             object
APELLIDO PATERNO            object
APELLIDO MATERNO            object
NOMBRES PROPIOS             object
PROCEDENCIA MP Y DEL        object
AVERIGUACION PREVIA         object
EXP SEMEFO                  object
FECHA                       object
EDAD                        object
SEXO                        object
DIAGNOSTICO                 object
BITACORA DE INGRESOS        object
PAGINA PDF                  object
FOJA                        object
OBSERVACIONES CAPTURISTA    object
TIPOS DE RESTOS             object
dtype: object

## Limpieza de datos

In [6]:
file_replace = "data_raw/SEMEFO_1974_correciones.csv"
dic_replace = pd.read_csv(file_replace)
dic_replace = create_hierarchical_catalog(dic_replace)
dic_replace

df = replace_by_column(df, dic_replace)

In [7]:
col_dtype = {
             'NUM PROG':'id', 'AVERIGUACION PREVIA':'id', 'EXP SEMEFO':'id', 
             'BITACORA DE INGRESOS':'id',  'PAGINA PDF':'id', 'FOJA':'int', 
             'NOMBRE COMPLETO':'str', 'APELLIDO PATERNO':'str', 'APELLIDO MATERNO':'str', 'NOMBRES PROPIOS':'str', 
             'OBSERVACIONES CAPTURISTA':'str', #'NOMBRE ALTERNATIVO':'str', 'POSIBLE IDENTIFICADO COMO':'str', 
             'SEXO':'categ', 'TIPOS DE RESTOS':'categ', 'PROCEDENCIA MP Y DEL':'categ', 'DIAGNOSTICO':'categ', 
             'EDAD':'int', 'FECHA':'datetime', 
            }

for col, dtype in col_dtype.items():
    print(col)
    if df[col].isnull().all():
        warn('Empty column {}'.format(col), stacklevel=2)
    else:
        df[col+'_clean'] = df[col].apply(detect_sd)
        if dtype=='id':
            df[col+'_clean'] = df[col+'_clean'].str.strip()
            df[col+'_clean'] = df[col+'_clean'].str.upper()
        if dtype=='str':
            df[col+'_clean'] = df[col+'_clean'].apply(simplify_string, str_case='upper')
        if dtype=='categ':
            pass
        if dtype=='int':
            if col=='EDAD':
                df[col+'_clean'] = df[col+'_clean'].apply(replace_edades_menor_al_año)
            df_astype = astype_and_separate_error(df[col+'_clean'], pd.to_numeric, downcast='integer')
            df[col+'_clean'] = df_astype[col+'_clean_astype']
            df[col+'_error'] = df_astype[col+'_clean_error']
        if dtype=='datetime':
            df_astype = astype_and_separate_error(df[col+'_clean'], pd.to_datetime, infer_datetime_format=True)
            df[col+'_clean'] = df_astype[col+'_clean_astype']
            df[col+'_error'] = df_astype[col+'_clean_error']
df

cols = flatten_list( [[c, c+'_clean', c+'_error'] for c in  all_cols] )
cols = [c for c in cols if c in df.columns]
df = df[cols]
df.to_csv(file_clean, index=False)
df

/home/esther/anaconda3/envs/ProfilesDatos/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3398: UserWarning: Empty column NUM PROG
  exec(code_obj, self.user_global_ns, self.user_ns)
/home/esther/anaconda3/envs/ProfilesDatos/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3398: UserWarning: Empty column BITACORA DE INGRESOS
  exec(code_obj, self.user_global_ns, self.user_ns)


NUM PROG
AVERIGUACION PREVIA
EXP SEMEFO
BITACORA DE INGRESOS
PAGINA PDF
FOJA
NOMBRE COMPLETO
APELLIDO PATERNO
APELLIDO MATERNO
NOMBRES PROPIOS
OBSERVACIONES CAPTURISTA
SEXO
TIPOS DE RESTOS
PROCEDENCIA MP Y DEL
DIAGNOSTICO
EDAD
FECHA


,NUM PROG,NOMBRE COMPLETO,NOMBRE COMPLETO_clean,APELLIDO PATERNO,APELLIDO PATERNO_clean,APELLIDO MATERNO,APELLIDO MATERNO_clean,NOMBRES PROPIOS,NOMBRES PROPIOS_clean,PROCEDENCIA MP Y DEL,...,BITACORA DE INGRESOS,PAGINA PDF,PAGINA PDF_clean,FOJA,FOJA_clean,FOJA_error,OBSERVACIONES CAPTURISTA,OBSERVACIONES CAPTURISTA_clean,TIPOS DE RESTOS,TIPOS DE RESTOS_clean
0,NaN,Arguello de Sandoval Elena,ARGUELLO DE SANDOVAL ELENA,Arguello,ARGUELLO,de Sandoval,DE SANDOVAL,Elena,ELENA,HB,...,NaN,1,1,3,3,NaN,NaN,NaN,Cadáver,Cadáver
1,NaN,Aragon Gómez David,ARAGON GOMEZ DAVID,Aragon,ARAGON,Gómez,GOMEZ,David,DAVID,HB,...,NaN,1,1,3,3,NaN,NaN,NaN,Cadáver,Cadáver
2,NaN,Arellano Sanchez Ana Maria,ARELLANO SANCHEZ ANA MARIA,Arellano,ARELLANO,Sanchez,SANCHEZ,Ana Maria,ANA MARIA,CR,...,NaN,1,1,3,3,NaN,NaN,NaN,Cadáver,Cadáver
3,NaN,Aguendez Martínez Arturo,AGUENDEZ MARTINEZ ARTURO,Aguendez,AGUENDEZ,Martínez,MARTINEZ,Arturo,ARTURO,3a,...,NaN,1,1,3,3,NaN,NaN,NaN,Cadáver,Cadáver
4,NaN,Alvarez Corona Rosa,ALVAREZ CORONA ROSA,Alvarez,ALVAREZ,Corona,CORONA,Rosa,ROSA,20a,...,NaN,1,1,3,3,NaN,NaN,NaN,Cadáver,Cadáver
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5728,NaN,R Nacido,R NACIDO,NaN,NaN,NaN,NaN,NaN,NaN,16a,...,NaN,150 reverso,150 REVERSO,219,219,NaN,R Nacido,R NACIDO,Cadáver,Cadáver
5729,NaN,R Nacido,R NACIDO,NaN,NaN,NaN,NaN,NaN,NaN,13a,...,NaN,150 reverso,150 REVERSO,219,219,NaN,R Nacido,R NACIDO,Cadáver,Cadáver
5730,NaN,R Nacido,R NACIDO,NaN,NaN,NaN,NaN,NaN,NaN,13a,...,NaN,150 reverso,150 REVERSO,219,219,NaN,R Nacido,R NACIDO,Cadáver,Cadáver
5731,NaN,R Nacido,R NACIDO,NaN,NaN,NaN,NaN,NaN,NaN,22a,...,NaN,150 reverso,150 REVERSO,219,219,NaN,R Nacido,R NACIDO,Cadáver,Cadáver


In [8]:
for col in df.columns:
    if 'error' in col:
        print(col)
        print( df[col].value_counts().index.to_list() )
        print('\n')

FECHA_error
[]


EDAD_error
[]


FOJA_error
[]




In [9]:
df = df.dropna(how='all', axis=1).dropna(how='all')
df.dtypes

NOMBRE COMPLETO                           object
NOMBRE COMPLETO_clean                     object
APELLIDO PATERNO                          object
APELLIDO PATERNO_clean                    object
APELLIDO MATERNO                          object
APELLIDO MATERNO_clean                    object
NOMBRES PROPIOS                           object
NOMBRES PROPIOS_clean                     object
PROCEDENCIA MP Y DEL                      object
PROCEDENCIA MP Y DEL_clean                object
AVERIGUACION PREVIA                       object
AVERIGUACION PREVIA_clean                 object
EXP SEMEFO                                object
EXP SEMEFO_clean                          object
FECHA                                     object
FECHA_clean                       datetime64[ns]
EDAD                                      object
EDAD_clean                               float64
SEXO                                      object
SEXO_clean                                object
DIAGNOSTICO         

## Ingeniería de datos

In [ ]:
from helper_pandas import translate_datetime

df['SEMANA'] = df['FECHA'].dt.isocalendar().week
df['DIA SEMANA'] = df['FECHA'].dt.day_name()
df['DIA SEMANA'] = translate_datetime(df['DIA SEMANA'])

from helper_pandas import divide_in_bins_for_age

i = df.columns.get_loc('EDAD')
df.insert(i+1, 'RANGO EDAD', divide_in_bins_for_age(df['EDAD'], step=5))
df['RANGO EDAD'] = df['RANGO EDAD'].replace('(5, 10]', '(05, 10]')


In [10]:
def palabras_cortas(s, thr=1):
    if type(s)==str:
        s = [i for i in s.split() if len(i)<=thr]
        s = ' '.join(s)
        if len(s)>0: return s
    return nan

df['NOMBRE COMPLETO_join'] = df[['APELLIDO PATERNO_clean','APELLIDO MATERNO_clean','NOMBRES PROPIOS_clean']] \
                               .apply(join_strings_with_nans, axis=1)
df['NOMBRE COMPLETO_limpio coincide'] = df['NOMBRE COMPLETO_clean']==df['NOMBRE COMPLETO_join']
df['TRANSCRIPCION INCIERTA'] = df['NOMBRE COMPLETO'].apply(punctuation_in_string, return_bool=True)
df['NOMBRES CORTO'] = df['NOMBRE COMPLETO_clean'].apply(palabras_cortas)
df['DESCONOCIDO'] = df['NOMBRE COMPLETO_clean'].str.contains('DESCONOCIDO')

In [11]:
df.to_csv(file_review, index=False)

## Profile

In [12]:
from pandas_profiling import ProfileReport

profile = ProfileReport(df, minimal=True)
profile.to_file(file_profi)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]